In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import flags
import numpy as np
import pandas as pd
pd.options.display.float_format = '{:.2f}'.format

In [3]:
df_vitals_combined_SBP_predicted = pd.read_csv('df_vitals_combined_SBP_predicted.csv')
df_vitals_combined_HR_predicted = pd.read_csv('df_vitals_combined_HR_predicted.csv')

In [4]:
df_vitals_combined_predicted = pd.merge(df_vitals_combined_SBP_predicted, df_vitals_combined_HR_predicted, on=['patient_uuid','reading_date'],how='inner')

In [5]:
df_vitals_combined = pd.read_csv('vitals_combined_interpolated.csv')

# Drops batches containing less than 30 observations.
df_recordings_per_patient = df_vitals_combined.groupby(['patient_uuid', 'batch']).count().reset_index()
low_batches = df_recordings_per_patient[(df_recordings_per_patient.HR <= 30)][['patient_uuid','batch']]

for i in range(len(low_batches)):
    low_patient = low_batches.patient_uuid.iloc[i]
    low_batch = low_batches.batch.iloc[i]
    df_vitals_combined.drop(df_vitals_combined[(df_vitals_combined.patient_uuid == low_patient) & (df_vitals_combined.batch == low_batch)].index, inplace=True)


print(df_vitals_combined.shape)
idx = df_vitals_combined.groupby(['patient_uuid'])['batch'].transform(max) == df_vitals_combined['batch']

df_vitals_combined = df_vitals_combined[idx]

print(df_vitals_combined.shape)

(121156, 5)
(111777, 5)


In [6]:
df_vitals_flags =  pd.DataFrame(columns = ['patient_uuid','HR_error','SBP_error','algo_accuracy','actual_rising_HR','actual_lowering_HR','actual_rising_SBP','actual_min_HR','actual_max_HR','actual_min_SBP','actual_max_SBP','actual_high_BP','pred_rising_HR','pred_lowering_HR','pred_rising_SBP','pred_min_HR','pred_max_HR','pred_min_SBP','pred_max_SBP','pred_high_BP'])
for patient_uuid in df_vitals_combined_predicted['patient_uuid'].unique():
    df_vitals_combined_patient_predicted = df_vitals_combined_predicted[df_vitals_combined_predicted['patient_uuid'] == patient_uuid]
    df_vitals_combined_patient_predicted.drop(columns=['patient_uuid'],inplace=True)
    df_vitals_combined_patient_predicted['reading_date'] = pd.to_datetime(df_vitals_combined_patient_predicted['reading_date'])
    df_vitals_combined_patient_predicted.sort_values(by='reading_date',inplace=True)
    df_vitals_combined_patient_predicted.set_index(keys='reading_date',inplace=True)    
    
    df_vitals_combined_patient = df_vitals_combined[df_vitals_combined['patient_uuid'] == patient_uuid]
    df_vitals_combined_patient.drop(columns=['patient_uuid','batch'],inplace=True)
    
    df_vitals_combined_patient_train = df_vitals_combined_patient.head(df_vitals_combined_patient.shape[0] - 7)
    df_vitals_combined_patient_train['reading_date'] = pd.to_datetime(df_vitals_combined_patient_train['reading_date'])
    df_vitals_combined_patient_train.sort_values(by='reading_date',inplace=True)
    df_vitals_combined_patient_train.set_index(keys='reading_date',inplace=True)

    df_vitals_combined_patient_test = df_vitals_combined_patient.tail(7)
    df_vitals_combined_patient_test['reading_date'] = pd.to_datetime(df_vitals_combined_patient_test['reading_date'])
    df_vitals_combined_patient_test.sort_values(by='reading_date',inplace=True)
    df_vitals_combined_patient_test.set_index(keys='reading_date',inplace=True)

    HR_error,SBP_error,algo_accuracy,actual_rising_HR,actual_lowering_HR,actual_rising_SBP,actual_min_HR,actual_max_HR,actual_min_SBP,actual_max_SBP,actual_high_BP,pred_rising_HR,pred_lowering_HR,pred_rising_SBP,pred_min_HR,pred_max_HR,pred_min_SBP,pred_max_SBP,pred_high_BP = \
    flags.compare_flags(df_vitals_combined_patient_train, df_vitals_combined_patient_test, df_vitals_combined_patient_predicted)
    
    df_vitals_flags.loc[len(df_vitals_flags)] = [patient_uuid,HR_error,SBP_error,algo_accuracy,actual_rising_HR,actual_lowering_HR,actual_rising_SBP,actual_min_HR,actual_max_HR,actual_min_SBP,actual_max_SBP,actual_high_BP,pred_rising_HR,pred_lowering_HR,pred_rising_SBP,pred_min_HR,pred_max_HR,pred_min_SBP,pred_max_SBP,pred_high_BP]
    
    #break

In [7]:
df_vitals_flags.to_csv("sarima_flags_stats.csv", index = False)

In [8]:
df_vitals_flags

,patient_uuid,HR_error,SBP_error,algo_accuracy,actual_rising_HR,actual_lowering_HR,actual_rising_SBP,actual_min_HR,actual_max_HR,actual_min_SBP,actual_max_SBP,actual_high_BP,pred_rising_HR,pred_lowering_HR,pred_rising_SBP,pred_min_HR,pred_max_HR,pred_min_SBP,pred_max_SBP,pred_high_BP
0,1127912,0.31,0.27,0.62,False,False,False,0,1,0,0,False,True,False,False,0,7,7,0,False
1,1164638,0.27,0.20,0.62,False,False,False,0,0,0,0,False,True,False,False,0,7,7,0,False
2,1166047,0.42,0.25,0.62,False,False,False,0,0,0,0,False,True,False,False,0,7,7,0,False
3,1193821,0.62,0.30,0.62,False,False,False,0,0,0,0,False,True,False,False,0,7,7,0,False
4,1231804,0.14,0.24,0.62,False,False,True,0,0,9,0,False,True,False,False,0,0,16,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
743,98613334,0.53,0.34,0.62,False,False,False,0,0,0,0,False,True,False,False,0,7,7,0,False
744,98660120,0.64,0.33,0.50,False,False,False,0,0,0,0,True,True,False,False,0,7,7,0,False
745,98716129,1.14,0.42,0.38,False,False,False,1,0,0,0,True,True,False,False,0,7,7,0,False
746,98999464,0.55,0.30,0.62,False,False,False,0,0,0,0,False,True,False,False,0,7,7,0,False


In [9]:
df_flags = df_vitals_flags[~(df_vitals_flags['SBP_error']>1)&~(df_vitals_flags['HR_error']>1)]

In [10]:

#rising hr
tp_rising_HR = np.sum((df_flags.pred_rising_HR == True) & (df_flags.actual_rising_HR == True))
tn_rising_HR = np.sum((df_flags.pred_rising_HR == False) & (df_flags.actual_rising_HR == False))
fp_rising_HR = np.sum((df_flags.pred_rising_HR == True) & (df_flags.actual_rising_HR == False))
fn_rising_HR = np.sum((df_flags.pred_rising_HR == False) & (df_flags.actual_rising_HR == True))


#lowering hr
tp_lowering_HR = np.sum((df_flags.pred_lowering_HR == True) & (df_flags.actual_lowering_HR == True))
tn_lowering_HR = np.sum((df_flags.pred_lowering_HR == False) & (df_flags.actual_lowering_HR == False))
fp_lowering_HR = np.sum((df_flags.pred_lowering_HR == True) & (df_flags.actual_lowering_HR == False))
fn_lowering_HR = np.sum((df_flags.pred_lowering_HR == False) & (df_flags.actual_lowering_HR == True))

# rising sbp
tp_rising_SBP = np.sum((df_flags.pred_rising_SBP == True) & (df_flags.actual_rising_SBP == True))
tn_rising_SBP = np.sum((df_flags.pred_rising_SBP == False) & (df_flags.actual_rising_SBP == False))
fp_rising_SBP = np.sum((df_flags.pred_rising_SBP == True) & (df_flags.actual_rising_SBP == False))
fn_rising_SBP = np.sum((df_flags.pred_rising_SBP == False) & (df_flags.actual_rising_SBP == True))


# high_bp
tp_high_BP = np.sum((df_flags.pred_high_BP == True) & (df_flags.actual_high_BP == True))
tn_high_BP = np.sum((df_flags.pred_high_BP == False) & (df_flags.actual_high_BP == False))
fp_high_BP = np.sum((df_flags.pred_high_BP == True) & (df_flags.actual_high_BP == False))
fn_high_BP = np.sum((df_flags.pred_high_BP == False) & (df_flags.actual_high_BP == True))

# metrics for binary flags
acc_rising_HR, f1_rising_HR = flags.accuracy_f1(tp_rising_HR, tn_rising_HR, fp_rising_HR, fn_rising_HR)
acc_lowering_HR, f1_lowering_HR = flags.accuracy_f1(tp_lowering_HR, tn_lowering_HR, fp_lowering_HR, fn_lowering_HR)
acc_rising_SBP, f1_rising_SBP = flags.accuracy_f1(tp_rising_SBP, tn_rising_SBP, fp_rising_SBP, fn_rising_SBP)
acc_high_BP, f1_high_BP = flags.accuracy_f1(tp_high_BP, tn_high_BP, fp_high_BP, fn_high_BP)

# mae for numerical flags
mae_min_HR = np.abs(df_flags.pred_min_HR - df_flags.actual_min_HR).mean()
mae_max_HR = np.abs(df_flags.pred_max_HR - df_flags.actual_max_HR).mean()
mae_min_SBP = np.abs(df_flags.pred_min_SBP - df_flags.actual_min_SBP).mean()
mae_max_SBP = np.abs(df_flags.pred_max_SBP - df_flags.actual_max_SBP).mean()


algo_metrics = [acc_rising_HR, f1_rising_HR, acc_lowering_HR, f1_lowering_HR, acc_rising_SBP, f1_rising_SBP,
                acc_high_BP, f1_high_BP, mae_min_HR, mae_max_HR, mae_min_SBP, mae_max_SBP]

algo_metrics_index = ['acc_rising_HR', 'f1_rising_HR', 'acc_lowering_HR', 'f1_lowering_HR', 'acc_rising_SBP',
                      'f1_rising_SBP','acc_high_BP', 'f1_high_BP', 'mae_min_HR', 'mae_max_HR', 'mae_min_SBP',
                      'mae_max_SBP']

algo_metrics

[0.01692524682651622,
 0.008534850640113799,
 0.9830747531734838,
 nan,
 0.8899858956276445,
 nan,
 0.8067700987306065,
 nan,
 0.12552891396332863,
 6.70098730606488,
 6.355430183356841,
 0.019746121297602257]